# XAI: Seg-Eigen-CAM
Change `RUN` below to explain a different model. Nothing else changes.

In [ ]:
%pip install -q grad-cam

In [ ]:
RUN = "02-segformer-mitb3-imagenet"

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation, SegformerConfig
from sklearn.model_selection import train_test_split

from pytorch_grad_cam import SegEigenCAM
from pytorch_grad_cam.utils.model_targets import SemanticSegmentationTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
ROOT = Path("../..")
MODEL_DIR = ROOT / "model" / RUN / "v1"
DATA = ROOT / "data" / "raw"
OUT = ROOT / "xai" / "out" / RUN
OUT.mkdir(parents=True, exist_ok=True)

metrics = json.load(open(MODEL_DIR / "test_metrics.json"))
CFG = metrics["cfg"]

CFG["arch"], CFG["init"], round(metrics["miou"], 4)

In [ ]:
labels_df = pd.read_csv(DATA / "Image_labels_Binary Classification Task.csv").rename(columns={
    "image name": "image_name",
    "Presence of plastic waste?": "presence"
})
reg_df = pd.read_csv(DATA / "Mask_foreground_percentages_Regression Task.csv").rename(columns={
    "image name": "image_name",
    "plastic waste accumulation (in percentage)": "fg_pct"
})

df = labels_df.merge(reg_df, on="image_name", how="inner")
df["image_path"] = df["image_name"].apply(lambda x: str(DATA / "Raw_Images" / x))
df["mask_path"] = df["image_name"].str.replace(".jpg", "_mask.png", regex=False).apply(lambda x: str(DATA / "Segmentation_Masks" / x))
df["has_plastic"] = (df["presence"].astype(str).str.lower() == "yes").astype(int)

train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=CFG["seed"], stratify=df["has_plastic"])
test_df = test_df.reset_index(drop=True)

len(test_df)

In [ ]:
def build_model(arch):
    if arch == "segformer":
        return SegformerForSemanticSegmentation(SegformerConfig.from_pretrained("nvidia/mit-b3", num_labels=2))
    return smp.Unet("mit_b3", encoder_weights=None, classes=2)


class Logits(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        if hasattr(out, "logits"):
            return F.interpolate(out.logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return out


net = build_model(CFG["arch"])
net.load_state_dict(torch.load(next(MODEL_DIR.glob("*.pth")), map_location="cpu", weights_only=True))
model = Logits(net).to(DEVICE).eval()

LAYER = net.decode_head.batch_norm if CFG["arch"] == "segformer" else net.decoder

S = CFG["img_size"]
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load(r):
    img = cv2.cvtColor(cv2.imread(r["image_path"]), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (S, S))
    gt = (cv2.imread(r["mask_path"], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    gt = cv2.resize(gt, (S, S), interpolation=cv2.INTER_NEAREST)
    x = torch.from_numpy(((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)).float()[None].to(DEVICE)
    return img, gt, x


@torch.no_grad()
def predict(x):
    return (torch.softmax(model(x), 1)[0, 1] > CFG["threshold"]).cpu().numpy().astype(np.uint8)

In [ ]:
def ious(p, t, eps=1e-7):
    tp = (p & t).sum()
    fp = (p & (1 - t)).sum()
    fn = ((1 - p) & t).sum()
    tn = ((1 - p) & (1 - t)).sum()
    return (tp + eps) / (tp + fp + fn + eps), (tn + eps) / (tn + fp + fn + eps)


rows = []
for _, r in test_df.iterrows():
    img, gt, x = load(r)
    pr = predict(x)
    fg, bg = ious(pr, gt)
    rows.append({"image_name": r["image_name"], "fg_iou": fg, "miou": (fg + bg) / 2,
                 "gt_pct": gt.mean() * 100, "pred_pct": pr.mean() * 100})

per_img = pd.DataFrame(rows)
per_img.to_csv(OUT / "per_image.csv", index=False)

print("reproduced", round(per_img["miou"].mean(), 4), "| training notebook", round(metrics["miou"], 4))

In [ ]:
class SegEigenCAMLite(SegEigenCAM):
    def get_cam_image(self, input_tensor, target_layer, targets, activations, grads, eigen_smooth):
        a = (np.abs(grads) * activations)[0]
        X = a.reshape(a.shape[0], -1).T
        X = X - X.mean(axis=0)
        _, _, VT = np.linalg.svd(X, full_matrices=False)
        p = (X @ VT[0]).reshape(a.shape[1:])
        if abs(p.min()) > abs(p.max()):
            p = -p
        return p[None]


cam = SegEigenCAMLite(model, [LAYER])


def explain(x, mask):
    return cam(x, [SemanticSegmentationTarget(1, mask.astype(np.float32))])[0]

In [ ]:
def show(names, title):
    fig, ax = plt.subplots(len(names), 5, figsize=(20, 4 * len(names)))
    for i, n in enumerate(names):
        img, gt, x = load(df[df.image_name == n].iloc[0])
        pr = predict(x)
        panels = [
            (img, n),
            (gt, f"truth {gt.mean() * 100:.2f}%"),
            (pr, f"pred {pr.mean() * 100:.2f}%"),
            (show_cam_on_image(img / 255.0, explain(x, pr), use_rgb=True), "why it predicted"),
            (show_cam_on_image(img / 255.0, explain(x, gt), use_rgb=True), "what it saw in truth"),
        ]
        for a, (im, t) in zip(ax[i], panels):
            a.imshow(im, cmap="gray")
            a.set_title(t)
            a.axis("off")
    fig.suptitle(f"{RUN}: {title}")
    plt.tight_layout()
    plt.savefig(OUT / f"{title}.png", dpi=90)
    plt.show()


show(["img1484.jpg", "img324.jpg", "img2.jpg", "img563.jpg"], "samples")

has = per_img[per_img.gt_pct > 0].sort_values("fg_iou")
show(list(has.image_name[:3]), "worst")
show(list(has.image_name[-3:]), "best")

In [ ]:
kept_hot, kept_cold = [], []

for n in has[has.pred_pct > 0].image_name:
    img, gt, x = load(df[df.image_name == n].iloc[0])
    pr = predict(x)
    c = explain(x, pr)

    idx = np.flatnonzero(pr)
    order = idx[np.argsort(c.flat[idx])]
    k = len(idx) // 2

    for chosen, kept in [(order[len(order) - k:], kept_hot), (order[:k], kept_cold)]:
        x2 = x.clone().flatten(2)
        x2[..., chosen] = 0
        kept.append(predict(x2.view_as(x))[pr == 1].mean())

summary = {"run": RUN, "test_miou": metrics["miou"], "n_images": len(kept_hot),
           "kept_after_hot_delete": float(np.mean(kept_hot)),
           "kept_after_cold_delete": float(np.mean(kept_cold)),
           "hot_hurts_more_in": int(sum(h < c for h, c in zip(kept_hot, kept_cold)))}
json.dump(summary, open(OUT / "xai_summary.json", "w"), indent=2)
summary